In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix
)
import matplotlib.pyplot as plt

from sklearn.neighbors import KNeighborsClassifier

In [2]:
np.random.seed(42)
n = 800

age             = np.random.randint(22, 65, n)
income          = np.random.normal(55000, 20000, n).clip(15000, 150000)
loan_amount     = np.random.normal(20000, 8000, n).clip(2000, 60000)
credit_score    = np.random.normal(680, 80, n).clip(300, 850)
emp_years       = np.random.randint(0, 30, n)
dti_ratio       = loan_amount / income          # Debt-to-income
num_credit_lines= np.random.randint(1, 10, n)

# Target: default probability depends on credit score, dti, income
log_odds = (
    -0.03 * (credit_score - 680)
    + 3.5  * dti_ratio
    - 0.01 * (income / 1000)
    + 0.02 * age
    - 0.05 * emp_years
    + np.random.normal(0, 0.5, n)
)
prob_default = 1 / (1 + np.exp(-log_odds))
default = (prob_default > 0.5).astype(int)

loan_df = pd.DataFrame({
    'age': age,
    'income': income.round(0),
    'loan_amount': loan_amount.round(0),
    'credit_score': credit_score.round(0),
    'emp_years': emp_years,
    'dti_ratio': dti_ratio.round(4),
    'num_credit_lines': num_credit_lines,
    'default': default
})

loan_df.head(2)

,age,income,loan_amount,credit_score,emp_years,dti_ratio,num_credit_lines,default
0,60,44390.0,14116.0,794.0,12,0.3180,2,0
1,50,43484.0,29889.0,754.0,12,0.6874,1,0


In [3]:
loan_df.shape

(800, 8)

In [4]:
loan_df.default.value_counts()/len(loan_df)

default
1    0.64
0    0.36
Name: count, dtype: float64

In [5]:
model = KNeighborsClassifier(n_neighbors=640)
model

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",640
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.Doesn't affect :meth:`fit` method.",None


In [6]:
X_lr = loan_df.drop('default', axis=1)
y_lr = loan_df['default']
X_lr

,age,income,loan_amount,credit_score,emp_years,dti_ratio,num_credit_lines
0,60,44390.0,14116.0,794.0,12,0.3180,2
1,50,43484.0,29889.0,754.0,12,0.6874,1
2,36,49499.0,28730.0,757.0,9,0.5804,1
3,64,15000.0,24873.0,779.0,17,1.6582,1
4,29,24696.0,11261.0,687.0,16,0.4560,5
...,...,...,...,...,...,...,...
795,47,36587.0,19174.0,735.0,22,0.5241,1
796,57,58387.0,22123.0,713.0,20,0.3789,4
797,22,26726.0,15338.0,655.0,3,0.5739,1
798,29,52775.0,2000.0,613.0,6,0.0379,4


In [7]:
X_train_lr, X_test_lr, y_train_lr, y_test_lr = train_test_split(
    X_lr, y_lr, test_size=0.2, random_state=42, stratify=y_lr
)

# y_train_lr.value_counts()/len(y_train_lr)
# y_test_lr.value_counts()/len(y_test_lr)
X_train_lr

,age,income,loan_amount,credit_score,emp_years,dti_ratio,num_credit_lines
664,42,87274.0,16278.0,725.0,16,0.1865,9
117,35,59964.0,22394.0,803.0,5,0.3735,1
730,55,59478.0,18695.0,651.0,21,0.3143,6
302,54,69155.0,26354.0,668.0,12,0.3811,7
398,44,60717.0,12006.0,619.0,5,0.1977,6
...,...,...,...,...,...,...,...
56,27,39735.0,20516.0,682.0,15,0.5163,7
473,23,72449.0,18649.0,579.0,2,0.2574,8
311,27,67413.0,13518.0,701.0,4,0.2005,2
34,28,27029.0,18600.0,610.0,27,0.6882,9


In [8]:
# feature scaling ?? you will not apply the formula --
scaler_lr = StandardScaler()
X_train_lr_sc = scaler_lr.fit_transform(X_train_lr)
X_test_lr_sc  = scaler_lr.transform(X_test_lr)

print(f"Train size : {X_train_lr_sc.shape}")
print(f"Test  size : {X_test_lr_sc.shape}")
print(f"Train default rate : {y_train_lr.mean()*100:.1f}%")
print(f"Test  default rate : {y_test_lr.mean()*100:.1f}%")


Train size : (640, 7)
Test  size : (160, 7)
Train default rate : 64.1%
Test  default rate : 63.7%


In [9]:
model.fit(X_train_lr_sc, y_train_lr)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",640
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.Doesn't affect :meth:`fit` method.",None


In [10]:
y_pred_lr   = model.predict(X_test_lr_sc)
y_pred_lr
y_proba_lr  = model.predict_proba(X_test_lr_sc)
# y_proba_lr[2]
y_proba_lr

array([[0.359375, 0.640625],
       [0.359375, 0.640625],
       [0.359375, 0.640625],
       [0.359375, 0.640625],
       [0.359375, 0.640625],
       [0.359375, 0.640625],
       [0.359375, 0.640625],
       [0.359375, 0.640625],
       [0.359375, 0.640625],
       [0.359375, 0.640625],
       [0.359375, 0.640625],
       [0.359375, 0.640625],
       [0.359375, 0.640625],
       [0.359375, 0.640625],
       [0.359375, 0.640625],
       [0.359375, 0.640625],
       [0.359375, 0.640625],
       [0.359375, 0.640625],
       [0.359375, 0.640625],
       [0.359375, 0.640625],
       [0.359375, 0.640625],
       [0.359375, 0.640625],
       [0.359375, 0.640625],
       [0.359375, 0.640625],
       [0.359375, 0.640625],
       [0.359375, 0.640625],
       [0.359375, 0.640625],
       [0.359375, 0.640625],
       [0.359375, 0.640625],
       [0.359375, 0.640625],
       [0.359375, 0.640625],
       [0.359375, 0.640625],
       [0.359375, 0.640625],
       [0.359375, 0.640625],
       [0.3593

In [11]:
acc  = accuracy_score(y_test_lr, y_pred_lr)

In [12]:
acc

# job is increase this acc_

0.6375

In [13]:
acc  = accuracy_score(y_test_lr, y_pred_lr)
prec = precision_score(y_test_lr, y_pred_lr)
rec  = recall_score(y_test_lr, y_pred_lr)
f1   = f1_score(y_test_lr, y_pred_lr)
print(f"  Accuracy  : {acc*100:.2f}%")
print(f"  Precision : {prec*100:.2f}%")
print(f"  Recall    : {rec*100:.2f}%")
print(f"  F1 Score  : {f1*100:.2f}%")

  Accuracy  : 63.75%
  Precision : 63.75%
  Recall    : 100.00%
  F1 Score  : 77.86%


In [45]:
# joib i s-- try different vaou of k 

k = [3,5,7,9,11,13]

for i in k:
    model = KNeighborsClassifier(n_neighbors=i)
    model.fit(X_train_lr_sc,y_train_lr)
    y_pred = model.predict(X_test_lr_sc)
    
    acc = accuracy_score(y_pred,y_test_lr)
    print('accuracy for k = {} is {}'.format(i,acc))


accuracy for k = 3 is 0.88125
accuracy for k = 5 is 0.875
accuracy for k = 7 is 0.875
accuracy for k = 9 is 0.88125
accuracy for k = 11 is 0.8875
accuracy for k = 13 is 0.8875
